In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("URL_Dataset.csv")

In [3]:
features = [
    'URLLength','DomainLength','IsDomainIP','TLDLength','NoOfSubDomain',
    'NoOfLettersInURL','LetterRatioInURL','NoOfDegitsInURL','DegitRatioInURL',
    'NoOfEqualsInURL','NoOfQMarkInURL','NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL','SpacialCharRatioInURL','IsHTTPS',
    'LineOfCode','LargestLineLength','HasTitle','HasFavicon',
    'NoOfiFrame','NoOfJS','NoOfImage','HasExternalFormSubmit',
    'HasHiddenFields','HasPasswordField','NoOfExternalRef','NoOfSelfRef'
]

In [4]:
X = df[features]
y = df['label']

# Fill missing values safely
X = X.fillna(0)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
#Reshaping for CNN
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

In [7]:
#building the CNN model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

model = Sequential([
    Conv1D(64, 3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(2),
    Conv1D(128, 3, activation='relu'),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\saish\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 25, 64)              │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 12, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 10, 128)             │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 189,057 (738.50 KB)

 Trainable params: 189,057 (738.50 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
#Training
history = model.fit(
    X_train_cnn, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test_cnn, y_test)
)

Epoch 1/15
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.9940 - loss: 0.0203 - val_accuracy: 0.9992 - val_loss: 0.0028
Epoch 2/15
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.9989 - loss: 0.0040 - val_accuracy: 0.9997 - val_loss: 0.0014
Epoch 3/15
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.9992 - loss: 0.0026 - val_accuracy: 0.9997 - val_loss: 0.0012
Epoch 4/15
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.9993 - loss: 0.0023 - val_accuracy: 0.9998 - val_loss: 9.1443e-04
Epoch 5/15
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.9996 - loss: 0.0017 - val_accuracy: 0.9996 - val_loss: 0.0013
Epoch 6/15
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.9994 - loss: 0.0017 - val_accuracy: 0.9997 - val_loss: 9.7236e-04
Epoch 7/15
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.9996 - loss: 0.0014 - val_accuracy: 0.9995 - val_loss: 0.0013
Epoch 8/15
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.9995 -

In [9]:
#evaluate
loss, accuracy = model.evaluate(X_test_cnn, y_test)
print("Test Accuracy:", accuracy)

1474/1474 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9998 - loss: 0.0010
Test Accuracy: 0.9998515844345093


In [10]:
#since the accuracy it 99, it's too perfect, tesing for any leaks,overfitting and duplicates
print(df['label'].value_counts())

label
1    134850
0    100945
Name: count, dtype: int64


In [11]:
df.duplicated().sum()

0

In [12]:
#checking feature correlation and abel

corr = df[features + ['label']].corr()
print(corr['label'].sort_values(ascending=False))

label                         1.000000
IsHTTPS                       0.609132
HasHiddenFields               0.507731
HasFavicon                    0.493711
HasTitle                      0.459725
NoOfJS                        0.373500
NoOfSelfRef                   0.316211
NoOfImage                     0.274658
LineOfCode                    0.272257
NoOfExternalRef               0.258627
NoOfiFrame                    0.225822
HasExternalFormSubmit         0.167574
HasPasswordField              0.138183
NoOfSubDomain                -0.005955
NoOfAmpersandInURL           -0.034622
LargestLineLength            -0.041111
IsDomainIP                   -0.060202
NoOfEqualsInURL              -0.076963
TLDLength                    -0.079159
NoOfQMarkInURL               -0.175621
NoOfDegitsInURL              -0.177980
URLLength                    -0.233445
NoOfLettersInURL             -0.258090
DomainLength                 -0.283152
NoOfOtherSpecialCharsInURL   -0.358891
LetterRatioInURL         

In [13]:
from sklearn.metrics import classification_report

y_pred = (model.predict(X_test_cnn) > 0.5).astype(int)

print(classification_report(y_test, y_pred))

1474/1474 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159



In [14]:
#building URL lexical feature extractor
import re
from urllib.parse import urlparse

def extract_url_features(url):
    features = {}

    parsed = urlparse(url)
    domain = parsed.netloc

    features['URLLength'] = len(url)
    features['DomainLength'] = len(domain)

    features['IsDomainIP'] = 1 if re.match(r'^\d+\.\d+\.\d+\.\d+$', domain) else 0

    tld = domain.split('.')[-1] if '.' in domain else ''
    features['TLDLength'] = len(tld)

    features['NoOfSubDomain'] = domain.count('.')

    features['NoOfLettersInURL'] = sum(c.isalpha() for c in url)
    features['LetterRatioInURL'] = features['NoOfLettersInURL'] / len(url)

    features['NoOfDegitsInURL'] = sum(c.isdigit() for c in url)
    features['DegitRatioInURL'] = features['NoOfDegitsInURL'] / len(url)

    features['NoOfEqualsInURL'] = url.count('=')
    features['NoOfQMarkInURL'] = url.count('?')
    features['NoOfAmpersandInURL'] = url.count('&')
    features['NoOfOtherSpecialCharsInURL'] = len(re.findall(r'[^\w]', url))
    features['SpacialCharRatioInURL'] = features['NoOfOtherSpecialCharsInURL'] / len(url)

    features['IsHTTPS'] = 1 if parsed.scheme == 'https' else 0

    return features

In [15]:
#building web scraper
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin

def extract_web_features(url):
    features = {
        'LineOfCode': 0,
        'LargestLineLength': 0,
        'HasTitle': 0,
        'HasFavicon': 0,
        'NoOfiFrame': 0,
        'NoOfJS': 0,
        'NoOfImage': 0,
        'HasExternalFormSubmit': 0,
        'HasHiddenFields': 0,
        'HasPasswordField': 0,
        'NoOfExternalRef': 0,
        'NoOfSelfRef': 0
    }

    try:
        response = requests.get(
            url,
            timeout=5,
            headers={'User-Agent': 'Mozilla/5.0'}
        )

        if response.status_code != 200:
            print("⚠ Non-200 status code:", response.status_code)
            return features

        html = response.text
        lines = html.split('\n')

        features['LineOfCode'] = len(lines)
        features['LargestLineLength'] = max(len(line) for line in lines) if lines else 0

        soup = BeautifulSoup(html, 'html.parser')

        # Title
        if soup.title and soup.title.string:
            features['HasTitle'] = 1

        # Favicon
        if soup.find('link', rel=lambda x: x and 'icon' in x.lower()):
            features['HasFavicon'] = 1

        # Tags count
        features['NoOfiFrame'] = len(soup.find_all('iframe'))
        features['NoOfJS'] = len(soup.find_all('script'))
        features['NoOfImage'] = len(soup.find_all('img'))

        # Password field
        if soup.find('input', {'type': 'password'}):
            features['HasPasswordField'] = 1

        # Hidden fields
        if soup.find('input', {'type': 'hidden'}):
            features['HasHiddenFields'] = 1

        # Forms external submit
        forms = soup.find_all('form')
        for form in forms:
            action = form.get('action')
            if action:
                action_url = urljoin(url, action)
                if urlparse(action_url).netloc != urlparse(url).netloc:
                    features['HasExternalFormSubmit'] = 1

        # Links analysis
        links = soup.find_all('a', href=True)
        for link in links:
            link_url = urljoin(url, link['href'])
            if urlparse(link_url).netloc != urlparse(url).netloc:
                features['NoOfExternalRef'] += 1
            else:
                features['NoOfSelfRef'] += 1

    except Exception as e:
        print("⚠ Scraping failed:", e)

    return features

In [16]:
#full prediction pipeline
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model

model = load_model("phishguard_cnn.h5")
scaler = joblib.load("scaler.pkl")
feature_columns = joblib.load("feature_columns.pkl")

def predict_url(url):
    url_feats = extract_url_features(url)
    web_feats = extract_web_features(url)

    combined = {**url_feats, **web_feats}

    # Create DataFrame to preserve feature names
    feature_df = pd.DataFrame([combined])

    # Ensure correct order
    feature_df = feature_df[feature_columns]

    # Scale
    scaled = scaler.transform(feature_df)

    # Reshape for CNN
    scaled = scaled.reshape(1, len(feature_columns), 1)

    # Predict
    prob = model.predict(scaled)[0][0]

    return {
        "URL": url,
        "Phishing Probability": float(prob),
        "Prediction": "Phishing" if prob > 0.5 else "Legitimate"
    }

In [17]:
print(predict_url("https://www.google.com"))
print(predict_url("http://testphp.vulnweb.com"))
print(predict_url("http://example.com"))
#here, the model seems to be works

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
{'URL': 'https://www.google.com', 'Phishing Probability': 0.0, 'Prediction': 'Legitimate'}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
{'URL': 'http://testphp.vulnweb.com', 'Phishing Probability': 0.0, 'Prediction': 'Legitimate'}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
{'URL': 'http://example.com', 'Phishing Probability': 0.0, 'Prediction': 'Legitimate'}
